# Easy allocation of projects to students
This notebook makes it easy to match students to projects, based on student preferences.

## Get it done with 4 steps
Create 2 csv files or amend them with actual data

1. `student_preferences.csv` with 1 + N preferences columns: student, preference1, preference2, ..., preferenceN. 👇Below is a preview of what the files should look like.
|student|preference1|preference2|preference3|preference4|preference5|
|:-----:|-----------|-----------|-----------|-----------|-----------|
|1	|Project1|	Project20|	Project3|	Project21	|Project2|
|2  |	Project10	|Project1|	Project4|	Project3|	Project9|


2. `project_capacities.csv` with 3 columns: project, capacity, supervisor.  👇Below is a preview of what the files should look like.
|project |capacity|supervisor|
|--------|-------:|---------:|
|Project1|1       |A         |
|Project2|1       |A         |


3. Save both files in this project directory.


4. Run the cell below to get the ideal matching after 16000 iterations.

In [ ]:
# THIS CELL RUNS THE WHOLE PROCESS!
# --------------------------------------------------------------------------------
# Load libraries and code
import pandas as pd
import time
import numpy as np
import multiprocessing
import easy_matching_functions as emf

if __name__ == "__main__":

    with multiprocessing.Manager() as manager:
        #  load data
        start = time.time()
        print("Finding best random seed from 16000 itterations... this may take a while!")
        print("-------------------------------------------------------------------------")
        student_preferences = pd.read_csv("student_preferences.csv")
        project_capacities = pd.read_csv("project_capacities.csv")
    
        results = manager.list()
        p1 = multiprocessing.Process(target=emf.find_best_seed_matching, args=(student_preferences, project_capacities, range(0, 2000), results))
        p2 = multiprocessing.Process(target=emf.find_best_seed_matching, args=(student_preferences, project_capacities, range(2000, 4000), results))
        p3 = multiprocessing.Process(target=emf.find_best_seed_matching, args=(student_preferences, project_capacities, range(4000, 6000), results))
        p4 = multiprocessing.Process(target=emf.find_best_seed_matching, args=(student_preferences, project_capacities, range(6000, 8000), results))
        p5 = multiprocessing.Process(target=emf.find_best_seed_matching, args=(student_preferences, project_capacities, range(8000, 10000), results))
        p6 = multiprocessing.Process(target=emf.find_best_seed_matching, args=(student_preferences, project_capacities, range(10000, 12000), results))
        p7 = multiprocessing.Process(target=emf.find_best_seed_matching, args=(student_preferences, project_capacities, range(12000, 14000), results))
        p8 = multiprocessing.Process(target=emf.find_best_seed_matching, args=(student_preferences, project_capacities, range(14000, 16000), results))
 
        p1.start()
        p2.start()
        p3.start()
        p4.start()
        p5.start()
        p6.start()
        p7.start()
        p8.start()

        p1.join()
        p2.join()
        p3.join()
        p4.join()
        p5.join()
        p6.join()
        p7.join()
        p8.join()

        best_seeds = list(results)

        df = pd.DataFrame(best_seeds, columns=["seed", "n_student_without_matching", "matching_rating"])
        df = df.sort_values(by=["n_student_without_matching", "matching_rating"], ascending=[True, False])
        end = time.time()
        print(f"Elapsed time: {end - start:.2f} seconds")
        print("-------------------------------------------------------------------------")
        print("These are the TOP seeds")
        print(df)
        top_seed = df.iloc[0,0]
        print("-------------------------------------------------------------------------")

        # Use optimal seed
        optimize_seed = top_seed
        np.random.seed(optimize_seed)

        students = []
        projects = []

        for index, row in student_preferences.iterrows():
            students.append(emf.Student(row["student"], row[1:].tolist()))

        for index, row in project_capacities.iterrows():
            projects.append(emf.Project(row["project"], row["capacity"]))

        np.random.shuffle(students)

        # Run the algorithm
        N_PREFERENCES_TO_CONSIDER = student_preferences.shape[1] - 1
        emf.gale_shapley(students, projects, N_PREFERENCES_TO_CONSIDER)

        print("==========")
        print(f"Using top seed {top_seed}")
        print(f'Final Matching score: {emf.rate_matching(students)}')
        emf.check_matching(students, projects) # check validity of matching

        # save as pandas
        matching_student_df = pd.DataFrame(columns=["student", "project", "rank_choice", "preference"])
        for i in range(len(students)):
            student = students[i]
            if student.matched_project is not None:
                matching_student_df.loc[i] = [student.id, student.matched_project.id, student.rank_choice, student.preferences]
            else: 
                matching_student_df.loc[i] = [student.id, None, 0, student.preferences]

            matching_project_df = pd.DataFrame(columns=["project", "students", "capacity"])
            for i in range(len(projects)):
                project = projects[i]
                matching_project_df.loc[i] = [project.id, [s.id for s in project.matched_students], project.capacity]

        # join project_capacities
        matching_project_df = matching_project_df.merge(project_capacities[["project","supervisor"]], left_on="project", right_on="project", how="left")

        # # export to csv
        matching_student_df.to_csv("matching_student.csv", index=False)
        matching_project_df.to_csv("matching_project.csv", index=False)
        print("-------------------------------------------------------------------------")
        print("The best matched results are saved to 2 files - same results different pespectives:") 
        print("    mattching_students.csv (Student perspective)")
        print("    matching_project.csv   (Project perspective)")
              
        